In [1]:
from keras.api.models import Model
from keras.api.models import load_model
import sentencepiece as spm
from pathlib import Path
import json
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
BASE_PATH = Path('/content/drive/MyDrive/AI/GMAv2.5/')
MODELS_PATH = BASE_PATH/'lstm_models'

In [ ]:
avalible_models = list(MODELS_PATH.glob('*/'))
avalible_models = [model.stem for model in avalible_models]
print(list(avalible_models))

In [ ]:
for i, model in enumerate(avalible_models, start=1):
    print(f"{i}) {model}")
model_index = input("Please select a model: ")
model_index = int(model_index)
if model_index < 1 or model_index > len(avalible_models):
    raise ValueError("Invalid model index")
model_name = avalible_models[model_index-1]
print(f"Selected model: {model_name}")

In [ ]:
model_config = MODELS_PATH/f"{model_name}/config.json"
with open(model_config) as f:
    model_config = json.load(f)
author_dictionary = model_config['author_dictionary']
tokenizer_name = model_config['tokenizer_name']
model_file = model_config['model_file']
model_file = MODELS_PATH/f"{model_name}/{model_file}"

model = load_model(model_file)
sp = spm.SentencePieceProcessor()
sp.load(str(BASE_PATH/tokenizer_name))

In [ ]:
message: str = input("Message: ")
ids = sp.encode_as_ids(message)
ids = np.array([ids])

predictions = model.predict(ids, verbose=0)
prediction = predictions[0]
auth_dict_censored = {f"{k[0]}{(len(k) - 2) * '*'}{k[-1]}":v for k,v in author_dictionary.items()}

for i, p in enumerate(prediction):
    print(f"{author_dictionary[str(i)]}: {p:.2f}")